<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ Watch The Book — Qwen3-TTS Headless SFT</h1>
  <h3 style='color: #cbd5e1; margin: 0 0 5px 0; font-weight: 400;'>Automated 1-Click SFT Training Pipeline</h3>
  <h5 style='color: #94a3b8; margin: 0 0 20px 0;'>Stable Single-Speaker Cloning | Automatic ASR & Formatting | T4 Optimized</h5>
  <div style='display: flex; justify-content: center; gap: 15px;'>
    <a href='https://www.youtube.com/@WatchTheBook?sub_confirmation=1' target='_blank' style='background: #3b82f6; color: white; padding: 10px 20px; border-radius: 8px; text-decoration: none; font-weight: bold;'>▶ YouTube</a>
  </div>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/Colab-Free%20Tier-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4_GPU_Enabled-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
</div>

---

### 🔗 Workspace Source Repository
This workspace runs on top of the official [jromal/qwen3-tts-voice-sft](https://github.com/jromal/qwen3-tts-voice-sft) training repository.

---


In [ ]:
# @title Step 1: ⚙️ Dynamic Configuration Dashboard
# Change the values inside the quotes below to customize your single-speaker fine-tuning run.
# After editing, you can run the notebook!

target_speaker_name = 'name_surname_en'
epochs_count = 'auto'         # Recommended: 'auto' (SFT: 720 / samples) or input an integer (e.g. 6) to override

# --- Dynamic Model Selection ---
# Choose model_scale ('1.7B' or '0.6B') — 'CustomVoice' is used to preserve pre-trained style attention weights
model_scale = '1.7B'
init_model_variant = f'Qwen/Qwen3-TTS-12Hz-{model_scale}-CustomVoice'

# --- Hugging Face Hub Integration ---
# Provide your HF Write Token and target repository path (e.g. 'username/repo-name')
push_to_hf = True
central_repo_id = 'user/repo'
hf_token = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

# =====================================================================
# SYSTEM EXECUTION CODE (Do not modify below this line)
# =====================================================================

# Save settings to a file so they persist across execution cells
import json
config_data = {
    'target_speaker_name': target_speaker_name,
    'epochs_count': epochs_count,
    'push_to_hf': push_to_hf,
    'init_model_variant': init_model_variant,
    'hf_token': hf_token,
    'central_repo_id': central_repo_id
}
with open('/tmp/sft_base_config.json', 'w') as f:
    json.dump(config_data, f)

print('==================================================')
print('⚙️ Configuration Registered Successfully!')
print(f'   👤 Speaker Name    : {target_speaker_name}')
print(f'   🤖 Init Model      : {init_model_variant}')
print(f'   ⏱️ Epochs          : {epochs_count}')
print(f'   📤 Push to HF Hub  : {push_to_hf}')
print(f'   📦 Target HF Repo  : {central_repo_id}')
print('==================================================')

In [ ]:
# @title Step 2: 🔍 Hardware Verification & Env Setup
import torch
import os

# Suppress non-fatal backend warnings and XLA factory registration noise
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ADJUST_HUGE_PAGES'] = '0'

# Prevent memory fragmentation during training forward passes
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Disable high-speed hf_transfer to prevent 403 Forbidden errors on Xet-bridge files
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

IS_GPU = torch.cuda.is_available()

if IS_GPU:
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'\n✅ GPU detected: {gpu_name}')
    print(f'✅ VRAM: {vram_total:.1f} GB')

if not IS_GPU:
    print('ℹ️ No GPU detected. Notebook will run on CPU mode (Standard Speed).')

In [ ]:
# @title Step 3: 📦 Install Headless Pipeline Dependencies
import os
print('📦 Synchronizing Audio and CLI SFT Environments...')

# 1. Install system-level ffmpeg, sox, development headers, and format codecs for high-fidelity audio calculations
!apt-get -y install ffmpeg sox libsox-dev libsox-fmt-all

# 2. Clone the official Qwen3-TTS upstream model repository
!git clone https://github.com/QwenLM/Qwen3-TTS.git

# 3. Switch Qwen3-TTS codebase to the designed commit hash
if os.path.exists('Qwen3-TTS'):
    os.chdir('Qwen3-TTS')
    !git checkout 0c6a7cbb6c8421a46332f8c2434c7825c4c855ef
    os.chdir('..')
else:
    print('❌ Error: Qwen3-TTS repository could not be cloned.')

# 4. Uninstall torchvision completely to prevent dynamic C++ binding crashes
print('🧹 Uninstalling torchvision to prevent dynamic C++ binding crashes...')
!pip uninstall torchvision -y

# 5. Install base dependencies, bitsandbytes, and runtime libraries
print('📥 Fetching companion requirements...')
!pip install qwen-asr hf_transfer accelerate transformers bitsandbytes peft onnxruntime einops

# 6. Overlay install qwen-tts using --no-deps to bypass version conflicts
!pip install qwen-tts --no-deps

print('\n✅ SFT Dependencies Installed Successfully!')

In [ ]:
%%writefile Qwen3-TTS/finetuning/sft_12hz.py
# coding=utf-8
import argparse
import json
import os
import shutil
from pathlib import Path
import torch
from accelerate import Accelerator
from dataset import TTSDataset
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from safetensors.torch import save_file
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoConfig

try:
    import torch.amp.grad_scaler
    _orig_unscale = torch.amp.grad_scaler.GradScaler._unscale_grads_
    torch.amp.grad_scaler.GradScaler._unscale_grads_ = lambda self, opt, inv, inf, allow=False: _orig_unscale(self, opt, inv, inf, True)
except Exception: pass

def get_attention_implementation():
    try:
        import flash_attn
        return 'flash_attention_2'
    except ImportError:
        return 'sdpa'

target_speaker_embedding = None
def train():
    global target_speaker_embedding
    parser = argparse.ArgumentParser()
    parser.add_argument('--init_model_path', type=str, default='Qwen/Qwen3-TTS-12Hz-1.7B-Base')
    parser.add_argument('--output_model_path', type=str, default='output')
    parser.add_argument('--train_jsonl', type=str, required=True)
    parser.add_argument('--batch_size', type=int, default=2)
    parser.add_argument('--lr', type=float, default=2e-6)
    parser.add_argument('--num_epochs', type=int, default=3)
    parser.add_argument('--speaker_name', type=str, default='speaker_test')
    args = parser.parse_args()

    device_cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
    target_dtype = torch.bfloat16 if device_cap[0] >= 8 else torch.float16

    logging_dir = os.path.join(args.output_model_path, 'logs')
    os.makedirs(logging_dir, exist_ok=True)
    grad_accum_steps = 4

    accelerator = Accelerator(
        gradient_accumulation_steps=grad_accum_steps,
        mixed_precision='bf16' if target_dtype == torch.bfloat16 else 'fp16',
        log_with='tensorboard',
        project_dir=logging_dir
    )

    MODEL_PATH = args.init_model_path
    attn_implementation = get_attention_implementation()
    qwen3tts = Qwen3TTSModel.from_pretrained(MODEL_PATH, dtype=target_dtype, attn_implementation=attn_implementation)

    # ── DYNAMIC SPEAKER ENCODER RESTORATION (Initializes with base config to prevent dimension mismatch) ──
    if not hasattr(qwen3tts.model, 'speaker_encoder') or qwen3tts.model.speaker_encoder is None:
        if accelerator.is_main_process:
            print('📥 CustomVoice base detected. Dynamically restoring Speaker Encoder from Base model...', flush=True)
        from qwen_tts.core.models.modeling_qwen3_tts import Qwen3TTSSpeakerEncoder
        from transformers import AutoConfig
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        try:
            # Target correct base config dynamically to match the model scale scale parameters
            base_model_id = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base' if '1.7B' in MODEL_PATH else 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
            base_config = AutoConfig.from_pretrained(base_model_id)
            
            # Instantiates with 2048-dim configurations to avoid CustomVoice 1024-dim mismatch failures
            qwen3tts.model.speaker_encoder = Qwen3TTSSpeakerEncoder(base_config.speaker_encoder_config)
            
            base_model_file = hf_hub_download(repo_id=base_model_id, filename='model.safetensors')
            base_state = load_file(base_model_file)
            encoder_state = {k.replace('speaker_encoder.', ''): v for k, v in base_state.items() if k.startswith('speaker_encoder.')}
            qwen3tts.model.speaker_encoder.load_state_dict(encoder_state)
            qwen3tts.model.speaker_encoder.to(device=qwen3tts.model.device, dtype=qwen3tts.model.dtype)
            if accelerator.is_main_process:
                print('✅ Speaker Encoder successfully restored for training.')
        except Exception as e:
            if accelerator.is_main_process:
                print(f'⚠️ Failed to restore Speaker Encoder: {e}', flush=True)

    if hasattr(qwen3tts.model, 'model') and hasattr(qwen3tts.model.model, 'gradient_checkpointing_enable'):
        qwen3tts.model.model.gradient_checkpointing_enable()
    elif hasattr(qwen3tts.model, 'gradient_checkpointing_enable'): qwen3tts.model.gradient_checkpointing_enable()
    config = AutoConfig.from_pretrained(MODEL_PATH)
    train_data = open(args.train_jsonl).readlines()
    train_data = [json.loads(line) for line in train_data]
    dataset = TTSDataset(train_data, qwen3tts.processor, config)
    train_dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, collate_fn=dataset.collate_fn)

    # FREEZE speaker_encoder AND speech_tokenizer to prevent voice scrambling
    if hasattr(qwen3tts.model, 'speaker_encoder') and qwen3tts.model.speaker_encoder is not None:
        for p in qwen3tts.model.speaker_encoder.parameters(): p.requires_grad = False
    for p in qwen3tts.model.parameters():
        if not any(name.startswith('talker') for name, _ in qwen3tts.model.named_parameters()):
            p.requires_grad = False
    for name, p in qwen3tts.model.named_parameters():
        if name.startswith('talker'):
            p.requires_grad = True

    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.PagedAdamW8bit(qwen3tts.model.talker.parameters(), lr=args.lr, weight_decay=0.01)
        optimizer_name = 'PagedAdamW8bit (bitsandbytes)'
    except ImportError:
        optimizer = AdamW(qwen3tts.model.talker.parameters(), lr=args.lr, weight_decay=0.01)
        optimizer_name = 'Standard AdamW (PyTorch)'

    model, optimizer, train_dataloader = accelerator.prepare(qwen3tts.model, optimizer, train_dataloader)
    num_epochs = args.num_epochs

    if accelerator.is_main_process:
        print('\\n' + '='*70)
        print('🎙️ QWEN3-TTS PATIENT FULL-PARAMETER SFT PIPELINE')
        print('='*70)
        print(f'👤 Speaker ID           : {args.speaker_name}')
        print(f'📊 Dataset Size (Samples): {len(dataset)}')
        print(f'📈 Effective Opt Steps  : {(len(train_dataloader) * num_epochs) // grad_accum_steps}')
        print(f'🎯 Attention Block     : {attn_implementation}')
        print(f'🧮 Hardware Precision  : {target_dtype}')
        print(f'⚙️  Only Training       : model.talker (Frozen encoder/vocoder)')
        print('='*70 + '\\n')

    model.train()
    for epoch in range(num_epochs):
        if hasattr(model, 'speaker_encoder') and model.speaker_encoder is not None:
            model.speaker_encoder.eval()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                input_ids = batch['input_ids']
                codec_ids = batch['codec_ids']
                ref_mels = batch['ref_mels']
                text_embedding_mask = batch['text_embedding_mask']
                codec_embedding_mask = batch['codec_embedding_mask']
                attention_mask = batch['attention_mask']
                codec_0_labels = batch['codec_0_labels']
                codec_mask = batch['codec_mask']

                speaker_embedding = model.speaker_encoder(ref_mels.to(model.device).to(model.dtype)).detach()
                if target_speaker_embedding is None:
                    target_speaker_embedding = speaker_embedding

                input_text_ids = input_ids[:, :, 0]
                input_codec_ids = input_ids[:, :, 1]

                # Dynamic Check: Avoid applying text_projection on 1.7B (fixes scrambled voice mismatch)
                raw_text_embedding = model.talker.model.text_embedding(input_text_ids)
                if raw_text_embedding.shape[-1] != model.talker.model.codec_embedding.weight.shape[-1]:
                    input_text_embedding = model.talker.text_projection(raw_text_embedding) * text_embedding_mask
                else:
                    input_text_embedding = raw_text_embedding * text_embedding_mask

                input_codec_embedding = model.talker.model.codec_embedding(input_codec_ids) * codec_embedding_mask
                input_codec_embedding[:, 6, :] = speaker_embedding
                input_embeddings = input_text_embedding + input_codec_embedding

                for i in range(1, 16):
                    codec_i_embedding = model.talker.code_predictor.get_input_embeddings()[i - 1](codec_ids[:, :, i])
                    codec_i_embedding = codec_i_embedding * codec_mask.unsqueeze(-1)
                    input_embeddings = input_embeddings + codec_i_embedding

                outputs = model.talker(inputs_embeds=input_embeddings, attention_mask=attention_mask, labels=codec_0_labels, output_hidden_states=True)
                hidden_states = outputs.hidden_states[0][-1]
                target_codec_mask = codec_mask[:, 1:]
                talker_hidden_states = hidden_states[:, :-1, :][target_codec_mask]
                talker_codec_ids = codec_ids[:, 1:][target_codec_mask]

                sub_talker_logits, sub_talker_loss = model.talker.forward_sub_talker_finetune(talker_codec_ids, talker_hidden_states)
                loss = outputs.loss + 0.3 * sub_talker_loss
                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

            if step % 10 == 0:
                accelerator.print(f'Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}')

        if accelerator.is_main_process:
            # ── PRE-PRUNING OLDER CHECKPOINTS ──
            # Deletes older checkpoints BEFORE saving the new one to strictly keep the Kaggle disk usage under 9.1 GB (Resolves quota error)
            output_path_obj = Path(args.output_model_path)
            saved_checkpoints = sorted(output_path_obj.glob('checkpoint-epoch-*'), key=lambda x: int(x.name.split('-epoch-')[-1]) if '-epoch-' in x.name else -1)
            for old_ckpt in saved_checkpoints:
                try:
                    shutil.rmtree(old_ckpt)
                    print(f'🧹 Pruned older checkpoint to free space: {old_ckpt.name}')
                except Exception: pass

            output_dir = os.path.join(args.output_model_path, f'checkpoint-epoch-{epoch}')
            from huggingface_hub import snapshot_download
            model_cache_path = MODEL_PATH if os.path.isdir(MODEL_PATH) else snapshot_download(MODEL_PATH)
            shutil.copytree(model_cache_path, output_dir, dirs_exist_ok=True)

            input_config_file = os.path.join(model_cache_path, 'config.json')
            output_config_file = os.path.join(output_dir, 'config.json')
            with open(input_config_file, 'r', encoding='utf-8') as f:
                config_dict = json.load(f)
            config_dict['tts_model_type'] = 'base'
            talker_config = config_dict.get('talker_config', {})
            talker_config['spk_id'] = {args.speaker_name: 3000}
            talker_config['spk_is_dialect'] = {args.speaker_name: False}
            config_dict['talker_config'] = talker_config
            with open(output_config_file, 'w', encoding='utf-8') as f:
                json.dump(config_dict, f, indent=2, ensure_ascii=False)

            unwrapped_model = accelerator.unwrap_model(model)
            state_dict = {k: v.detach().to('cpu').to(torch.bfloat16) for k, v in unwrapped_model.state_dict().items()}
            weight = state_dict['talker.model.codec_embedding.weight']
            state_dict['talker.model.codec_embedding.weight'][3000] = target_speaker_embedding[0].detach().to(weight.device).to(weight.dtype)
            save_file(state_dict, os.path.join(output_dir, 'model.safetensors'))
            print(f'✅ Saved updated model weights to: {output_dir}')

if __name__ == '__main__':
    train()

In [ ]:
# @title Step 4: 🚀 Ingest Dataset & Run Headless SFT Training

import os
import re
import glob
import json
import shutil
from pathlib import Path
import torch
from tqdm import tqdm

# Ensure directory alignment safely using Python's native os.chdir
working_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
if os.path.exists(working_dir):
    os.chdir(working_dir)
print(f'📂 Active Working Directory: {os.getcwd()}')

# Load configurations from dashboard
with open('/tmp/sft_base_config.json', 'r') as f:
    config = json.load(f)
target_speaker_name = config['target_speaker_name']
epochs_count = config['epochs_count']
init_model_variant = config['init_model_variant']

# Create standard local output directory
output_dir = Path('output')
output_dir.mkdir(parents=True, exist_ok=True)

# ── 1. DYNAMIC DATASET RESOLUTION & INGESTION ──
input_root = Path('/kaggle/input')
print(f'🔍 Scanning {input_root} on disk...')

jsonl_src = None
ref_src = None
ref_txt_src = None
audio_src_dir = None

# Recursively find your dataset files inside /kaggle/input/
if input_root.exists():
    for path in input_root.rglob('train_raw.jsonl'):
        jsonl_src = path
        break
    for path in input_root.rglob('ref.wav'):
        ref_src = path
        break
    for path in input_root.rglob('ref.txt'):
        ref_txt_src = path
        break

if not jsonl_src:
    raise ValueError('❌ Error: Could not locate \'train_raw.jsonl\' in your attached Kaggle datasets. Please ensure you have attached your dataset correctly.')

print(f'✅ Found transcription file: {jsonl_src}')

# Read the first line of the JSONL to extract your custom folder structure
with open(jsonl_src, 'r', encoding='utf-8') as f:
    first_line = json.loads(f.readline().strip())

# Extract the custom audio path (e.g., './wavs/utt0001.wav' -> 'wavs')
audio_path_in_jsonl = first_line.get('audio', '')
path_parts = Path(audio_path_in_jsonl).parts
if len(path_parts) > 1:
    target_sub_folder = path_parts[-2]
else:
    target_sub_folder = ''

if target_sub_folder == '.':
    target_sub_folder = ''

# Locate the source folder in /kaggle/input matching that target subfolder name
if target_sub_folder:
    for path in input_root.rglob(target_sub_folder):
        if path.is_dir():
            audio_src_dir = path
            break
else:
    audio_src_dir = jsonl_src.parent

if not audio_src_dir:
    raise ValueError(f'❌ Error: Could not locate a directory named \'{target_sub_folder}\' in your attached datasets to match your JSONL pathing.')

print(f'✅ Found audio directory: {audio_src_dir}')

# Copy files locally maintaining the exact folder structure your JSONL expects
local_workspace = Path('.')
shutil.copy2(jsonl_src, local_workspace / 'train_raw.jsonl')

if ref_src:
    shutil.copy2(ref_src, local_workspace / 'ref.wav')
    print('✅ Copied reference file \'ref.wav\' to workspace.')

if ref_txt_src:
    shutil.copy2(ref_txt_src, local_workspace / 'ref.txt')
    print('✅ Copied reference transcript file \'ref.txt\' to workspace.')

# Create the local subfolder and copy all wav files there
if target_sub_folder:
    local_subfolder = local_workspace / target_sub_folder
    local_subfolder.mkdir(parents=True, exist_ok=True)
    print(f'📁 Replicating local folder: {local_subfolder}')
    for wav_path in audio_src_dir.glob('*.wav'):
        shutil.copy2(wav_path, local_subfolder / wav_path.name)
    print(f'✅ Copied {len(list(local_subfolder.glob("*.wav")))} audio files to {local_subfolder}/')
else:
    # Flat copy
    for wav_path in audio_src_dir.glob('*.wav'):
        shutil.copy2(wav_path, local_workspace / wav_path.name)
    print('✅ Copied audio files flat to workspace.')

# ── 1.2 DYNAMIC EPOCHS CALCULATION ──
num_samples = len(list(local_subfolder.glob('*.wav'))) if target_sub_folder else len(list(local_workspace.glob('*.wav')))
print(f'📊 Dataset Audit: Found {num_samples} valid audio samples.')

if epochs_count == 'auto':
    # Apply standard Full SFT formula (720 / num_samples) capped cleanly between 4 and 20 epochs
    calculated_epochs = round(720 / max(num_samples, 1))
    epochs_count = min(max(calculated_epochs, 4), 20)
    print(f'📊 Auto-Epoch Engine: Dynamically set epochs_count = {epochs_count}')
else:
    epochs_count = int(epochs_count)
    print(f'📊 Manual Override: Training strictly for {epochs_count} epochs as requested.')

# Update config file with the actual calculated epochs so Step 5 can read it cleanly
config['epochs_count'] = epochs_count
with open('/tmp/sft_base_config.json', 'w') as f:
    json.dump(config, f)

# ── 1.5. FLUSH HUGGINGFACE CACHE TO FREE DISK SPACE ──
print('🧹 Flushing Hugging Face cache to ensure enough disk space is free...')
hf_hub_cache = Path('/root/.cache/huggingface/hub')
if hf_hub_cache.exists():
    for p in hf_hub_cache.glob('*'):
        if p.is_dir():
            try:
                shutil.rmtree(p)
            except Exception:
                pass

# ── 2. EXTRACT CODEC IDS (PREPARE DATA) ──
print('🎙️ Extracting audio codes via prepare_data.py...')
!python Qwen3-TTS/finetuning/prepare_data.py \
    --device cuda:0 \
    --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
    --input_jsonl train_raw.jsonl \
    --output_jsonl train_with_codes.jsonl
    
print('✅ Successfully prepared data with codec IDs.')

# ── 2.5 NOTEBOOK PARAMETERS PREPARATION BLOCK ──
print("\\n" + "="*70)
print("🚀 READY TO COMMENCE HEADLESS SFT TRAINING PIPELINE")
print("="*70)
print(f"   👤 Speaker Directory Label : {target_speaker_name}")
print(f"   🤖 Selected Base Model     : {init_model_variant}")
print(f"   📊 Input Audio Count       : {num_samples} audio samples")
print(f"   ⏱️  SFT Target Epoch Count  : {epochs_count}")
print(f"   ⚡ Learning Rate           : 2e-6")
print(f"   🔋 Device Batch Size       : 1")
print(f"   🔄 Gradient Accumulation   : 4")
print("="*70 + "\\n")

# ── 3. EXECUTE HEADLESS SFT TRAINING PIPELINE ──
print('🚀 Starting Headless Qwen3-TTS Full SFT Training (Batch Size: 1, FP16/BF16 adaptive)...\\n')
!python Qwen3-TTS/finetuning/sft_12hz.py \
    --init_model_path {init_model_variant} \
    --train_jsonl train_with_codes.jsonl \
    --output_model_path output \
    --batch_size 1 \
    --lr 2e-6 \
    --num_epochs {epochs_count} \
    --speaker_name {target_speaker_name}

In [ ]:
# @title Step 5: 📤 Push Trained Model to Hugging Face Hub
# This automatically uploads your trained single-speaker checkpoints directly to your centralized repository.

import json
import os
from pathlib import Path

# Ensure directory alignment safely using Python's native os.chdir pointing to root workspace
working_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
if os.path.exists(working_dir):
    os.chdir(working_dir)
print(f'📂 Current Working Directory: {os.getcwd()}')

# Load configuration from dashboard
with open('/tmp/sft_base_config.json', 'r') as f:
    config = json.load(f)
target_speaker_name = config['target_speaker_name']
epochs_count = config['epochs_count']
push_to_hf = config.get('push_to_hf', True)
hf_token = config.get('hf_token', '')
central_repo_id = config.get('central_repo_id', '')

output_dir = os.path.abspath('output')

# Strictly target the final planned epoch folder to avoid saving unfinished checkpoints
final_epoch_folder_name = f'checkpoint-epoch-{epochs_count - 1}'
upload_target_dir = os.path.join(output_dir, final_epoch_folder_name)

# For Full SFT, we verify that the model weights and config exist under upload_target_dir
final_model_file = os.path.join(upload_target_dir, 'model.safetensors')
final_config_file = os.path.join(upload_target_dir, 'config.json')

if push_to_hf and os.path.exists(final_model_file) and os.path.exists(final_config_file):
    # ── REDUNDANT FILES PURGER ──
    # Deletes unnecessary speech-tokenizer subfolders while strictly preserving text tokenizer configuration files.
    # Full SFT checkpoints require 'vocab.json', 'merges.txt', and 'tokenizer_config.json' to remain loadable via from_pretrained.
    print('🧹 Pruning redundant directories before uploading...')
    import shutil
    shutil.rmtree(os.path.join(upload_target_dir, 'speech_tokenizer'), ignore_errors=True)
    
    # We only delete the preprocessor config for the speech components, keeping the core text processor files.
    for redundant_file in ['preprocessor_config_speech_tokenizer.json']:
        f_path = os.path.join(upload_target_dir, redundant_file)
        if os.path.exists(f_path):
            os.remove(f_path)

    # ── COPY REFERENCE AUDIO & TRANSCRIPT ASSETS TO CHECKPOINT ──
    # Injects both ref.wav and ref.txt into the Hugging Face folder with dual-naming support to enable ICL voice cloning on the server
    local_ref_wav = os.path.join(working_dir, 'ref.wav')
    local_ref_txt = os.path.join(working_dir, 'ref.txt')
    
    if os.path.exists(local_ref_wav):
        shutil.copy2(local_ref_wav, os.path.join(upload_target_dir, 'ref.wav'))
        shutil.copy2(local_ref_wav, os.path.join(upload_target_dir, 'ref_sample.wav'))
        print('✅ Successfully copied ref.wav / ref_sample.wav reference files to output checkpoint.')
        
    if os.path.exists(local_ref_txt):
        shutil.copy2(local_ref_txt, os.path.join(upload_target_dir, 'ref.txt'))
        shutil.copy2(local_ref_txt, os.path.join(upload_target_dir, 'ref_sample.txt'))
        print('✅ Successfully copied ref.txt / ref_sample.txt reference transcript files to output checkpoint.')
            
    print(f'📤 Preparing to upload final completed epoch folder {upload_target_dir} directly to central HF repository: {central_repo_id}/{target_speaker_name}...')
    try:
        from huggingface_hub import HfApi, login, create_repo
        login(token=hf_token)
        api = HfApi()
        
        # Create the central repository if it does not exist (skip if active)
        create_repo(repo_id=central_repo_id, exist_ok=True, repo_type='model')
        
        # ── ZERO-HISTORY BRANCH RECREATION ──
        # Deletes the branch if it already exists to completely clear out all historical LFS cache footprints
        print(f'🧹 Deleting existing branch "{target_speaker_name}" on Hugging Face to clear LFS history...')
        try:
            api.delete_branch(
                repo_id=central_repo_id,
                branch=target_speaker_name,
                repo_type='model'
            )
            print('   ✅ Remote branch deleted.')
        except Exception:
            print('   ℹ️ Branch does not exist yet (or is already clean). Skipping deletion.')

        # ── EXPLICIT BRANCH RE-INITIALIZATION (Fixes 404 Revision Not Found) ──
        print(f'🌿 Initializing fresh, clean branch "{target_speaker_name}" on the Hub...')
        api.create_branch(
            repo_id=central_repo_id,
            branch=target_speaker_name,
            repo_type='model'
        )
        print('   ✅ Remote branch created successfully.')

        # Upload the final epoch folder files directly as an independent branch (automatically creates the branch)
        print(f'📤 Uploading clean, finalized model weights and configurations to branch: "{target_speaker_name}"...')
        api.upload_folder(
            folder_path=upload_target_dir,
            repo_id=central_repo_id,
            revision=target_speaker_name,
            repo_type='model'
        )
        print(f'🎉 Success! Your voice is saved directly at: https://huggingface.co/{central_repo_id}/tree/{target_speaker_name}')
    except Exception as e:
        print(f'❌ Failed to push model to Hugging Face: {e}')

if not push_to_hf:
    print('ℹ️ Hugging Face upload is disabled in configuration.')

if push_to_hf and (not os.path.exists(final_model_file) or not os.path.exists(final_config_file)):
    print(f'❌ Error: Final completed model weights (model.safetensors / config.json) not found in {upload_target_dir}.')
    print('   Training was not fully completed to the last planned epoch. Aborting upload to prevent saving unfinished models.')